## All necessary imports for this notebook

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path("../src").resolve()))

In [4]:
import json
import subprocess
from PIL import Image
import numpy as np
import fitz
import torch
from types import SimpleNamespace
from dataset import AlignCollate
from model import Model
from huggingface_hub import hf_hub_download
from utils import CTCLabelConverter
from llama_cpp import Llama

In [5]:
# Resolve project root to avoid machine-specific absolute paths.
# The notebook assumes it is located inside the repository (e.g. notebooks/).
# Paths to data, models, and outputs are constructed relative to this root.

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

# Rendering a PDF page

Use ```render_pdf_page``` for rendering a single PDF page. Inputs are: 
1. ```pdf_path```: path to the PDF file.
2. ```page_num```: number of the PDF page to render.
1. ```out_dir```: path to store the rendered page.
1. ```dpi```: render DPI.

In [6]:

def render_pdf_page(pdf_path, page_num, out_dir="tmp", dpi=400):
    """
    Renders a single page from a PDF to a PNG image.
    Args:
        pdf_path (str or Path): Path to the PDF file.
        page_num (int): 1-based page number to render.
        out_dir (str or Path): Directory to save the image.
        dpi (int): Render DPI.
    Returns:
        Path to the saved PNG image.
    """
    pdf_path = Path(pdf_path).expanduser().resolve()
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    scale = dpi / 72.0
    mat = fitz.Matrix(scale, scale)
    with fitz.open(pdf_path) as doc:
        if not (1 <= page_num <= doc.page_count):
            raise ValueError(f"Page number {page_num} out of range (1-{doc.page_count})")
        page = doc.load_page(page_num - 1)
        pix = page.get_pixmap(matrix=mat, alpha=False)
        out_path = out_dir / f"{pdf_path.stem}_page{page_num}.png"
        pix.save(str(out_path))
    return out_path

In [ ]:
# Example usage:
pdf_file = DATA_DIR / "Print" / "Covarrubias - Tesoro lengua.pdf"  # Set your PDF path here
page_number = 7  # Set the page number you want to render
img_path = render_pdf_page(pdf_file, page_number)
print(f"Saved page {page_number} as image: {img_path}")

## Line Segmentation

Once the PDF page has been rendered as an image, line segmentation is performed using **Kraken**.  
This step separates the tasks of **line detection** and **text transcription**, which allows the OCR model to operate on individual text lines.

Each detected line is cropped and saved as an independent `.png` image. These line images are then used as inputs for the OCR recognition model.

In [8]:
def run_kraken_segment(png_path, out_json, text_direction='horizontal-lr'):
    cmd = [
        'kraken',
        '-i', str(png_path),
        str(out_json),
        'segment',
        '-bl',
        '-d', text_direction,
    ]
    subprocess.run(cmd, check=True)

def bbox_from_boundary(boundary):
    pts = np.asarray(boundary, dtype=np.float32)
    xs = pts[:, 0]
    ys = pts[:, 1]
    return int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())

def extract_lines_from_kraken_json(seg):
    lines = seg.get('lines', [])
    out = []
    for i, ln in enumerate(lines):
        bbox = None
        if isinstance(ln, dict):
            if 'bbox' in ln and ln['bbox'] is not None:
                bb = ln['bbox']
                if isinstance(bb, (list, tuple)) and len(bb) == 4:
                    bbox = tuple(map(int, bb))
            elif 'boundary' in ln and ln['boundary'] is not None:
                bbox = bbox_from_boundary(ln['boundary'])
        if bbox is None:
            continue
        out.append({'i': i, 'bbox': bbox, 'raw': ln})
    return out

In [9]:
# --- Segment page and save crops ---
page_path = PROJECT_ROOT / "notebooks" / "tmp" / "Covarrubias - Tesoro lengua_page7.png"
output_dir = PROJECT_ROOT / "notebooks" / "output_lines"
output_dir.mkdir(exist_ok=True)
seg_json = output_dir / 'segmentation.json'

run_kraken_segment(page_path, seg_json)
seg = json.loads(seg_json.read_text(encoding='utf-8'))
lines = extract_lines_from_kraken_json(seg)

page_img = Image.open(page_path).convert('L')
crops = []
for i, ln in enumerate(lines, start=1):
    x1, y1, x2, y2 = ln['bbox']
    crop = page_img.crop((x1, y1, x2, y2))
    crops.append((crop, f'line_{i:04d}'))  # label is dummy

Loading ANN /home/tomas/projects/renAIssance/.venv/lib/python3.10/site-packages/kraken/blla.mlmodel	✓
Segmenting /home/tomas/projects/renAIssance/notebooks/tmp/Covarrubias - Tesoro lengua_page7.png	✓


In [10]:
# --- Build a simple dataset and dataloader using AlignCollate ---
class SimpleLineDataset(torch.utils.data.Dataset):
    def __init__(self, crops):
        self.crops = crops
    def __len__(self):
        return len(self.crops)
    def __getitem__(self, idx):
        img, label = self.crops[idx]
        return img, label

opt = type('opt', (), {})()
opt.imgH = 64  # Set to your model's training height

collate_fn = AlignCollate(imgH=opt.imgH)
dataset = SimpleLineDataset(crops)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)

## Load Pretrained Model

A pretrained OCR model is loaded for inference.

A trained checkpoint is available in the repository releases. Download the checkpoint and place it in the `models/` directory of the project.

For details about the model architecture, training procedure, and dataset preparation, refer to the [`README.md`](../README.md) file in the repository.

In [11]:
# --- Load checkpoint ---
ckpt_path = PROJECT_ROOT / "models" / "OCR_model.ckpt"
ckpt = torch.load(ckpt_path, map_location="cpu")

# --- Recover training hyperparameters ---
hparams = ckpt["hyper_parameters"]
opt = SimpleNamespace(**hparams)

converter = CTCLabelConverter(opt.character)
opt.num_class = len(converter.character)

# --- Prepare state dictionary ---
state_dict = ckpt["state_dict"]
state_dict = {k.replace("model.", ""): v for k, v in state_dict.items()}

# --- Instantiate model and load weights ---
model = Model(opt)
model.load_state_dict(state_dict)

model.eval()

Model(
  (FeatureExtraction): ResNet_FeatureExtractor(
    (ConvNet): ResNet(
      (conv0_1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn0_1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv0_2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn0_2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): BasicBlock(
          (conv1): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=Tru

## Run OCR Inference

The pretrained model is applied to the segmented line images to generate text predictions.  
The decoded transcriptions are printed and saved to a `.txt` file.

In [12]:
# Output file
output_txt_path = PROJECT_ROOT / "notebooks" / "OCR_outputs" / "predictions.txt"
output_txt_path.parent.mkdir(parents=True, exist_ok=True)

predictions = []
device = next(model.parameters()).device

with torch.no_grad():
    for images, _ in dataloader:

        images = images.to(device)
        batch_size = images.size(0)

        preds = model(images)

        _, preds_index = preds.max(2)
        preds_size = torch.IntTensor([preds.size(1)] * batch_size)

        preds_str = converter.decode(preds_index, preds_size)

        for pred in preds_str:
            print(pred)
            predictions.append(pred)

# Save predictions
with open(output_txt_path, "w", encoding="utf-8") as f:
    f.write("\n".join(predictions))

print(f"Predictions saved to {output_txt_path}")

E  h V C aa C-
LaC-
PEDRDE VRLENCIA CRRNiTA-
General de el Rey nuestro-
Señor.-
1or mandedo del Consejo Supremo de Casrilla, he visto el libro in-
titulado,rzsoro de la lengza Castellana, ue compuso el Ficen-
ciado Don Sebastian de Cobarrcuras y rozco, Canonigo, y Maesses-
cuela de la Iglesia de Cuenca, Consultor de el Santo Oticio de la Inquisi-
ción, y Capelan de su Magestad, y no he hallado en cl cosa contraria a-
la Cz, ni a las bsenas costzmDrzs; antzs riene mucñas muy vtiles, p está lle
mo de varia, 3 curiosa leccion, y doctrina. Por lo qual, y por la autoridad,-
y erudicion de la perfsona del Autor, tan conocida, y estimada entodas par-
tes, y porquede materia semejnnte han escrito en cada lengca, y nacion-
politica, varones m?y graves, y doctos, y por ser conveniente que de la-
propiedad, purtza, y zlegancia de vna lengua se zscriva én zl tiempo que-
ellamasfsorze, meparece se devedar la licencia, y privilegio que se pde pa
traimprimirlo. Fn Madrid a tres diasdel mes de Mayo de

## LLM-based Post-processing

To further improve the OCR output, we apply a **large language model (LLM)** to post-process the predicted text lines. The LLM is used to correct likely transcription errors by considering the **linguistic context of the sentence**, which can help resolve ambiguities produced by the OCR model.

The post-processing step uses the [Qwen2.5-3B-Instruct](https://huggingface.co/Qwen/Qwen2.5-3B-Instruct) model, an open-weight language model developed by the Qwen team. The model is executed locally using the [llama.cpp](https://github.com/ggml-org/llama.cpp) inference framework.

The model weights are downloaded from the [Hugging Face Hub](https://huggingface.co/), a widely used platform for sharing machine learning models and datasets.

In this pipeline, the LLM receives the OCR predictions together with a simple system prompt. More complex and detailed prompts were used, including notes taken by experts over the original prints, but for a lighweight model simpler prompts proved to get better results. The model then produces a corrected version of the text while preserving the intended reading of the original document. The system prompt is written in Spanish, as the text being processed is also in that language.

In this implementation, the LLM generates text without imposing formatting constraints relative to the OCR input. Empirically, this approach produced more coherent corrections. On some runs, the output respects the line structure on the input; on others, the LLM produces continuous text. With larger or more capable language models, it may be possible to obtain structured outputs that preserve the original line segmentation while maintaining similar correction quality.

In [7]:
model_path = hf_hub_download(
    repo_id="Qwen/Qwen2.5-3B-Instruct-GGUF",
    filename="qwen2.5-3b-instruct-q4_k_m.gguf",
)

llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_threads=8,
)

llama_model_loader: loaded meta data with 26 key-value pairs and 435 tensors from /home/tomas/.cache/huggingface/hub/models--Qwen--Qwen2.5-3B-Instruct-GGUF/snapshots/7dabda4d13d513e3e842b20f0d435c732f172cbe/qwen2.5-3b-instruct-q4_k_m.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = qwen2.5-3b-instruct
llama_model_loader: - kv   3:                            general.version str              = v0.1-v0.1
llama_model_loader: - kv   4:                           general.finetune str              = qwen2.5-3b-instruct
llama_model_loader: - kv   5:                         general.size_label str              = 3.4B
llama_model_loade

In [23]:
SYSTEM_PROMPT = """Estás asistiendo en la corrección posterior a OCR de textos históricos en español.

Recibirás un texto extraído por OCR a partir de líneas consecutivas de una página. El texto puede contener errores de reconocimiento, grafías antiguas y líneas irrelevantes.

Tu tarea es reconstruir la lectura más probable del texto original y devolverlo en español moderno. Debes corregir los errores de las predicciones del OCR, 
corrigiendo palabras mal formadas o mal reconocidas, unir palabras partidas por saltos de línea cuando sea razonable, y elminar líneas que no tengan ningún sentido.

Convenciones a tener en cuenta:
- Las letras "u" y "v" pueden aparecer intercambiadas. Interprétalas según la ortografía moderna.
- Las letras "f" y "s" pueden aparecer intercambiadas. Interprétalas según la ortografía moderna.
- Los acentos son inconsistentes. Ignóralos excepto la "ñ", que debe conservarse.
- Algunas letras presentan una pequeña barra horizontal. Esto suele indicar que sigue una "n", o que tras una "q" sigue "ue".
- La "ç" debe interpretarse siempre como "z" en español moderno.

Devuelve únicamente el texto corregido, es español moderno, sin comentarios ni explicaciones adicionales.
"""

In [27]:
SYSTEM_PROMPT = """
Vas a recibir un texto extraído por OCR con numerosas erratas, escrito en un español incorrecto. Tu tarea es corregir el texto, reconstruyendo la lectura más probable del texto original.
Devuelve únicamente el texto corregido, sin comentarios ni explicaciones adicionales. Si alguna palabra no eres capaz de corregirla, escribe [?] en su lugar.
"""

In [28]:
# Join OCR predictions into one page text
page_text = "\n".join(predictions)

# Call the LLM
response = llm.create_chat_completion(
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Texto OCR:\n\n{page_text}"}
    ],
    temperature=0.0
)

corrected_text = response["choices"][0]["message"]["content"].strip()


# Show result in notebook
print("\n\n----- LLM OUTPUT -----\n")
print(corrected_text)


# Save result
output_llm_path = PROJECT_ROOT / "notebooks" / "LLM_outputs" / "predictions_llm.txt"
output_llm_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_llm_path, "w", encoding="utf-8") as f:
    f.write(corrected_text)

print(f"\nLLM output saved to {output_llm_path}")

Llama.generate: 67 prefix-match hit, remaining 393 prompt tokens to eval
llama_perf_context_print:        load time =   19798.47 ms
llama_perf_context_print: prompt eval time =    9365.04 ms /   393 tokens (   23.83 ms per token,    41.96 tokens per second)
llama_perf_context_print:        eval time =   26494.79 ms /   289 runs   (   91.68 ms per token,    10.91 tokens per second)
llama_perf_context_print:       total time =   36439.42 ms /   682 tokens
llama_perf_context_print:    graphs reused =        279




----- LLM OUTPUT -----

La C-
PEDRO DE URVENCIA CRISTIANA
General del Rey nuestro-Senor.
Por mandato del Consejo Supremo de Casrilla, he visto el libro
título "Río de la lengua Castellana", que compuso el Filósofo Don
Sebastian de Cobarrubias y Rozco, Canonigo, y Maestre de la Iglesia
de Cuenca, Consultor del Santo Oticio de la Inquisición, y Capelán de
su Magestad, y no he hallado en él cosa contraria a la C, ni a las
benes costumdrzas; aunque contiene muchas muy útiles, y curiosa
leccion, y doctrina. Por lo cual, y por la autoridad, y erudición de la
persona del Autor, tan conocida y estimada en todas partes, y por
que de materia semejante han escrito en cada lengua y nación política,
varones más graves y doctos, y por ser conveniente que de la propiedad,
pertenencia, y lengua se escriba en el tiempo que las letras florecen, me
parece deber la licencia y privilegio que se pueda para imprimirlo.
En Madrid a tres días del mes de Mayo de 1610.
Oál le Malenía.
-

LLM output saved to /h